In [ ]:
import numpy as np
import pandas as pd

from hedgingEnvironment import evaluatePolicy
from agents import MeanStdDDPGConfig, MeanStdDDPGAgent, trainDDPG, makeReinforcementLearningPolicy
from datahandling import preprocess_kaggle_data
from sanity_checks import prechecks
from itertools import product

In [2]:
print(prechecks())

All checks passed.


In [ ]:
results = {}

train_eps = 800
eval_eps = 300

r = "daily"
m = "3m"
d = "rl"

tauVals = np.arange(0.0001, 0.001, 0.0005)
alrVals = np.arange(0.0001, 0.001, 0.0005)
clrVals = np.arange(0.00004, 0.0004, 0.0002)
nstdVals = np.arange(0.005, 0.05, 0.02)
nclipVals = np.arange(0.02, 0.2, 0.08)
rwsVals = [1, 50, 100]

paramCombos = product(tauVals, alrVals, clrVals, nstdVals, nclipVals, rwsVals)
for param_combo in paramCombos:
    paperSpecTest = EnvSpec(kappaMode="paper", maturity=m, rebalancingFrequency=r, kappaPaper=0.01)
    envFunc = lambda spec=paperSpecTest: makeEnvironment(spec)
    env = envFunc()
    state_dim = len(env.stateVector())
    config = MeanStdDDPGConfig()
    config.tau, config.actorLearningRate, config.criticLearningRate, config.noiseStd, config.noiseClip = param_combo[:-1]
    agent = MeanStdDDPGAgent(state_dim, config)
    trainDDPG(envFunc, agent, episodes=train_eps, debugFirstEpisode=False, logEvery=np.inf, reward_scalar=param_combo[-1])
    rl_policy = makeReinforcementLearningPolicy(agent)
    summary, df = evaluatePolicy(envFunc, rl_policy, episodes=eval_eps)
    results[(d, m, r, *param_combo)] = (summary["meanCostPct"], summary["stdCostPct"], summary["Y(mean+c*std)"])
    cols = ['delta', 'maturity', 'rebal', 'tau', 'alr', 'clr', 'nstd', 'nclip', 'rws', 'mean_cost', 'std_cost', 'Y_metric']
    df = pd.DataFrame([(*k, *v) for k, v in results.items()], columns=cols)
    df.to_csv("results_paper_0.01.csv", index=False)

In [ ]:
df = pd.read_csv("results_paper_0.01.csv")
top_25_params = df.sort_values(by="Y_metric", ascending=True).head(25).iloc[:, 3:-3].reset_index(drop=True)
params = list(top_25_params.itertuples(index=False, name=None))

r = "daily"
m = "3m"
d = "rl"

for mode, kappa in [("paper", 0.05), ("stochastic", 0.01)]:
    results = {}
    for param_combo in params:
        paperSpecTest = EnvSpec(kappaMode=mode, maturity=m, rebalancingFrequency=r, kappaPaper=kappa)
        envFunc = lambda spec=paperSpecTest: makeEnvironment(spec)
        env = envFunc()
        state_dim = len(env.stateVector())
        config = MeanStdDDPGConfig()
        config.tau, config.actorLearningRate, config.criticLearningRate, config.noiseStd, config.noiseClip = param_combo[:-1]
        agent = MeanStdDDPGAgent(state_dim, config)
        trainDDPG(envFunc, agent, episodes=train_eps, debugFirstEpisode=False, logEvery=np.inf, reward_scalar=param_combo[-1])
        rl_policy = makeReinforcementLearningPolicy(agent)
        summary, df = evaluatePolicy(envFunc, rl_policy, episodes=eval_eps)
        results[(d, m, r, *param_combo)] = (summary["meanCostPct"], summary["stdCostPct"], summary["Y(mean+c*std)"])
    cols = ['delta', 'maturity', 'rebal', 'tau', 'alr', 'clr', 'nstd', 'nclip', 'rws', 'mean_cost', 'std_cost', 'Y_metric']
    df = pd.DataFrame([(*k, *v) for k, v in results.items()], columns=cols)
    
    if mode == "stochastic":
        df.to_csv("results_stochastic.csv", index=False)
    else:
        df.to_csv(f"results_{mode}_{kappa}.csv", index=False)

In [ ]:
df = pd.read_csv("results_paper_0.01.csv")
top_25_params = df.sort_values(by="Y_metric", ascending=True).head(25).iloc[:, 3:-3].reset_index(drop=True)
params = list(top_25_params.itertuples(index=False, name=None))

r = "daily"
m = "3m"
d = "rl"

if RUN_HEAVY_TOP25:
    for mode, kappa in [("paper", 0.05), ("stochastic", 0.01)]:
        results = {}
        for param_combo in params:
            paperSpecTest = EnvSpec(kappaMode=mode, maturity=m, rebalancingFrequency=r, kappaPaper=kappa)
            envFunc = lambda spec=paperSpecTest: makeEnvironment(spec)
            env = envFunc()
            state_dim = len(env.stateVector())
            config = MeanStdDDPGConfig()
            config.tau, config.actorLearningRate, config.criticLearningRate, config.noiseStd, config.noiseClip = param_combo[:-1]
            agent = MeanStdDDPGAgent(state_dim, config)
            trainDDPG(envFunc, agent, episodes=train_eps, debugFirstEpisode=False, logEvery=np.inf, reward_scalar=param_combo[-1])
            rl_policy = makeReinforcementLearningPolicy(agent)
            summary, df = evaluatePolicy(envFunc, rl_policy, episodes=eval_eps)
            results[(d, m, r, *param_combo)] = (summary["meanCostPct"], summary["stdCostPct"], summary["Y(mean+c*std)"])
        cols = ['delta', 'maturity', 'rebal', 'tau', 'alr', 'clr', 'nstd', 'nclip', 'rws', 'mean_cost', 'std_cost', 'Y_metric']
        df = pd.DataFrame([(*k, *v) for k, v in results.items()], columns=cols)

        if mode == "stochastic":
            df.to_csv("results_stochastic.csv", index=False)
        else:
            df.to_csv(f"results_{mode}_{kappa}.csv", index=False)
else:
    print("FAST_SUBMISSION_MODE: skipping top-25 retraining, using existing results_paper_0.05.csv / results_stochastic.csv")

## 13) Take the params that had the highest averages, select them as model candidates

In [ ]:
df1 = pd.read_csv("results_paper_0.01.csv")
df5 = pd.read_csv("results_paper_0.05.csv")
dfsto = pd.read_csv("results_stochastic.csv")
df_first2 = df1.merge(df5, on=["delta", "maturity", "rebal", "tau", "alr", "clr", "nstd", "nclip", "rws"], suffixes=("_0.01_paper", "_0.05_paper"))
df_final = df_first2.merge(dfsto, on=["delta", "maturity", "rebal", "tau", "alr", "clr", "nstd", "nclip", "rws"])
df_final = df_final.rename(columns = {"mean_cost": "mean_cost_stochastic", "std_cost": "std_cost_stochastic", "Y_metric": "Y_metric_stochastic"})
df_final["Y_metric_avg"] = df_final[["Y_metric_0.01_paper", "Y_metric_0.05_paper", "Y_metric_stochastic"]].mean(axis=1)
df_final = df_final.sort_values(by="Y_metric_avg", ascending=True).reset_index(drop=True)
df_final.to_csv("merged_results.csv", index=False)
model_3 = df_final[["tau", "alr", "clr", "nstd", "nclip", "rws", "Y_metric_0.01_paper", "Y_metric_0.05_paper", "Y_metric_stochastic", "Y_metric_avg"]].head(3)
model_3

## 14) Re-training the top 3 models for longer & compare to baseline strategies

In [ ]:
params = [("delta",), ("delta", 0.25)]
params += [row[:-4] for row in model_3.itertuples(index=False, name=None)]
results = {}

if RUN_HEAVY_TOP3:
    for mode, kappa in [("paper", 0.01), ("paper", 0.05), ("stochastic", None)]:
        print(f"Evaluating top models for mode={mode}, kappa={kappa}")
        for param_combo in params:
            if len(param_combo) <= 2:
                paperSpecTest = EnvSpec(kappaMode=mode, maturity=m, rebalancingFrequency=r, kappaPaper=kappa)
                envFunc = lambda spec=paperSpecTest: makeEnvironment(spec)
                if param_combo == ("delta",):
                    policy = policyDeltaHedge
                else:
                    policy = policyDeltaHedgeWithBand(band=param_combo[1])
                summary, df = evaluatePolicy(envFunc, policy, episodes=300)
            else:
                tau, alr, clr, nstd, nclip, rws = param_combo[:6]
                paperSpecTest = EnvSpec(kappaMode=mode, maturity=m, rebalancingFrequency=r, kappaPaper=kappa)
                envFunc = lambda spec=paperSpecTest: makeEnvironment(spec)
                env = envFunc()
                state_dim = len(env.stateVector())
                config = MeanStdDDPGConfig()
                config.tau, config.actorLearningRate, config.criticLearningRate, config.noiseStd, config.noiseClip = tau, alr, clr, nstd, nclip
                agent = MeanStdDDPGAgent(state_dim, config)
                trainDDPG(envFunc, agent, episodes=1000, debugFirstEpisode=False, logEvery=np.inf, reward_scalar=rws)
                rl_policy = makeReinforcementLearningPolicy(agent)
                summary, df = evaluatePolicy(envFunc, rl_policy, episodes=300)

            print(f"Params: {param_combo} | meanCostPct: {summary['meanCostPct']:.4f} | stdCostPct: {summary['stdCostPct']:.4f} | Y_metric: {summary['Y(mean+c*std)']:.4f}")
            results[(mode, kappa, *param_combo)] = (summary["meanCostPct"], summary["stdCostPct"], summary["Y(mean+c*std)"])
else:
    print("FAST_SUBMISSION_MODE: skipping top-3 long retraining. Will load final_table.parquet in later cell.")

In [ ]:
fastMode = globals().get("FAST_SUBMISSION_MODE", False)

if fastMode and os.path.exists("final_table.parquet"):
    df = pd.read_parquet("final_table.parquet")
    df
else:
    assert "results" in globals(), ("results not found. Run the top-3 retraining cell above first, or enable FAST_SUBMISSION_MODE and keep final_table.parquet in the directory.")
    assert "model_3" in globals(), "`model_3` not found. Run the model-selection cell (cell 32) first."
    assert len(model_3) >= 3, "model_3 must contain at least 3 rows."

    param_map = {i + 1: (float(model_3.iloc[i]["tau"]), float(model_3.iloc[i]["alr"]), float(model_3.iloc[i]["clr"]),
                         float(model_3.iloc[i]["nstd"]), float(model_3.iloc[i]["nclip"]), int(model_3.iloc[i]["rws"]))
                 for i in range(3)}

    requiredKeys = [("paper", 0.01, "delta"), ("paper", 0.05, "delta"), ("stochastic", None, "delta"),
                    ("paper", 0.01, "delta", 0.25), ("paper", 0.05, "delta", 0.25), ("stochastic", None, "delta", 0.25)]
    for i in [1, 2, 3]:
        requiredKeys += [("paper", 0.01, *param_map[i]), ("paper", 0.05, *param_map[i]), ("stochastic", None, *param_map[i])]
    for key in requiredKeys:
        assert key in results, f"Missing results key: {key}"

    def r(x):
        return tuple(round(float(v), 1) for v in x)

    df = pd.DataFrame(index=["Delta", "DeltaWithBands", "Model #1", "Model #2", "Model #3"],
                      columns=["Constant-0.01", "Constant-0.05", "Stochastic"],
                      dtype=object)

    # Baselines
    df.loc["Delta", "Constant-0.01"] = r(results[("paper", 0.01, "delta")])
    df.loc["Delta", "Constant-0.05"] = r(results[("paper", 0.05, "delta")])
    df.loc["Delta", "Stochastic"]    = r(results[("stochastic", None, "delta")])

    df.loc["DeltaWithBands", "Constant-0.01"] = r(results[("paper", 0.01, "delta", 0.25)])
    df.loc["DeltaWithBands", "Constant-0.05"] = r(results[("paper", 0.05, "delta", 0.25)])
    df.loc["DeltaWithBands", "Stochastic"]    = r(results[("stochastic", None, "delta", 0.25)])

    # RL models
    for i in [1, 2, 3]:
        df.loc[f"Model #{i}", "Constant-0.01"] = r(results[("paper", 0.01, *param_map[i])])
        df.loc[f"Model #{i}", "Constant-0.05"] = r(results[("paper", 0.05, *param_map[i])])
        df.loc[f"Model #{i}", "Stochastic"]    = r(results[("stochastic", None, *param_map[i])])

    s = df.stack()
    expanded = pd.DataFrame(s.tolist(), index=s.index, columns=["Avg Cost", "Std Cost", "Y Metric"])
    df = expanded.unstack(level=1)
    df.columns = df.columns.swaplevel(0, 1)
    df = df.sort_index(axis=1, level=0)

    # Average Y metric across regimes and sort best-to-worst
    df[("", "Avg Y Metric")] = df.xs("Y Metric", axis=1, level=1).mean(axis=1).round(1)
    df.sort_values(("", "Avg Y Metric"), ascending=True, inplace=True)

    assert not df.isna().any().any(), "Final simulation summary table contains NaNs"

    df.to_parquet("final_table.parquet")
    df

In [ ]:
df

# 15) Defining tweaked environment to handle real data

In [ ]:
import numpy as np
from itertools import cycle

class GroupedDataHedgingEnv(hedgingEnvironment):
    """
    Uses historical paths from a grouped DataFrame.
    Required columns: '[UNDERLYING_LAST]' '[MID_PRICE]' '[DTE]'
    """

    def __init__(self, group, **kwargs):

        group = group.sort_values("[QUOTE_DATE]").reset_index(drop=True)

        steps = len(group) - 1
        S0 = float(group["[UNDERLYING_LAST]"].iloc[0])

        super().__init__(S0=S0, steps=steps, **kwargs)

        self.group = group
        self.S_path = self.group["[UNDERLYING_LAST]"].values
        self.V_path = -self.options * self.group["[MID_PRICE]"].values
        self.tau_path = self.group["[DTE]"].values / 365.0

        # Safety check
        assert np.all(np.diff(self.group["[DTE]"]) <= 0), "DTE must be non-increasing"

    def tau(self):
        return float(self.tau_path[self.i])

    def reset(self):
        self.i = 0
        self.S = float(self.S_path[0])

        if self.startFromStationary:
            self.L = int(self.rng.choice(3, p=self.pi))
        else:
            self.L = self.startingRegime

        self.kappat = float(self.kappaLevels[self.L])
        self.H = 0
        self.V = float(self.V_path[0])

        return self.stateVector(), 0.0

    def step(self, Hnext):

        Hnext = float(np.clip(Hnext, self.Hmin, self.Hmax))

        kappaUsed = float(self.kappat)
        regimeUsed = int(self.L)

        Snext = float(self.S_path[self.i + 1])
        Vnext = float(self.V_path[self.i + 1])

        transactionCost = kappaUsed * abs(Snext * (Hnext - self.H))

        reward = (Vnext - self.V) + self.H * (Snext - self.S) - transactionCost

        self.i += 1
        self.S, self.H, self.V = Snext, Hnext, Vnext

        done = (self.i >= self.N)
        terminalTC = 0

        if done:
            terminalTC = kappaUsed * abs(self.S * self.H)
            reward -= terminalTC
        else:
            self.L = markovRegimeTransition(self.L, self.P, self.rng)
            self.kappat = float(self.kappaLevels[self.L])

        info = {"TransactionCost": transactionCost, "TotalTransactionCost": transactionCost + terminalTC,
                "TerminalTransactionCost": terminalTC, "KappaUsed": kappaUsed, "RegimeUsed": regimeUsed
               }
        return self.stateVector(), float(reward), done, info
    

def makeGroupedEnvironment(spec):
    if spec.data is None or len(spec.data) == 0:
        raise ValueError("spec.data has no groups. Check preprocess_kaggle_data filters / input parquet files.")
    
    if not hasattr(spec, "_group_cycle"):
        spec._group_cycle = cycle(spec.data)

    group_key, group_df = next(spec._group_cycle)

    if spec.kappaMode == "paper":
        kappaLevels = [spec.kappaPaper, spec.kappaPaper, spec.kappaPaper]
        P = np.eye(3)
    elif spec.kappaMode == "stochastic":
        kappaLevels = list(spec.kappaLevelsStoch)
        P = spec.PStoch if spec.PStoch is not None else DEFAULTREGIMETRANSITION
    else:
        raise ValueError("kappaMode must be 'paper' or 'stochastic'")

    return GroupedDataHedgingEnv(group=group_df, K=group_key[1], T=group_df["[DTE]"].iloc[0] / 365.0, kappaLevels=kappaLevels, P=P)

# 16) Importing Kaggle data and comparing candidate models and baselines on real world data

In [ ]:
if "RUN_HEAVY_REALDATA" not in globals():
    RUN_HEAVY_REALDATA = not globals().get("FAST_SUBMISSION_MODE", False)

hist = []

if RUN_HEAVY_REALDATA:
    assert "model_3" in globals(), "`model_3` not found. Run the model-selection cell first."
    assert os.path.exists("spy_2020_2022.parquet"), "Missing file: spy_2020_2022.parquet"
    assert os.path.exists("qqq_2020_2022.parquet"), "Missing file: qqq_2020_2022.parquet"

    params = [("delta",), ("delta", 0.25)]
    params += [row[:-4] for row in model_3.itertuples(index=False, name=None)]  # top-3 RL params only
    results = {}

    spyData = pd.read_parquet("spy_2020_2022.parquet")
    qqqData = pd.read_parquet("qqq_2020_2022.parquet")

    mode = "paper"
    kappa = 0.05
    r = "daily"
    m = "3m"

    for assetName, assetDf in [("spy", spyData), ("qqq", qqqData)]:
        print(assetName)
        groupsTrain, groupsTest = preprocess_kaggle_data(assetDf)
        print(len(groupsTrain), len(groupsTest))

        for paramCombo in params:
            print(f"Evaluating params: {paramCombo}")

            # Baselines: delta and delta-with-band
            if len(paramCombo) <= 2:
                paperSpecEval = EnvSpec(kappaMode=mode, maturity=m, rebalancingFrequency=r, kappaPaper=kappa, data=groupsTest)
                envFuncEval = lambda spec=paperSpecEval: makeGroupedEnvironment(spec)

                if paramCombo == ("delta",):
                    policy = policyDeltaHedge
                else:
                    policy = policyDeltaHedgeWithBand(band=paramCombo[1])

                summary, evalDf = evaluatePolicy(envFuncEval, policy, episodes=len(groupsTest))
                results[(assetName, *paramCombo)] = (summary["meanCostPct"], summary["stdCostPct"], summary["Y(mean+c*std)"], evalDf)

            # RL models: train on train groups, evaluate on test groups
            else:
                tau, alr, clr, nstd, nclip, rws = paramCombo[:6]
                paperSpecTrain = EnvSpec(kappaMode=mode, maturity=m, rebalancingFrequency=r, kappaPaper=kappa, data=groupsTrain)
                envFuncTrain = lambda spec=paperSpecTrain: makeGroupedEnvironment(spec)
                envTrain = envFuncTrain()

                stateDim = len(envTrain.stateVector())
                config = MeanStdDDPGConfig()
                config.tau = tau
                config.actorLearningRate = alr
                config.criticLearningRate = clr
                config.noiseStd = nstd
                config.noiseClip = nclip

                agent = MeanStdDDPGAgent(stateDim, config)

                # Training
                hist.append(trainDDPG(envFuncTrain, agent, episodes=int(min(len(groupsTrain), 300)),
                                      debugFirstEpisode=False, logEvery=np.inf, reward_scalar=rws))

                # Testing
                paperSpecEval = EnvSpec(kappaMode=mode, maturity=m, rebalancingFrequency=r, kappaPaper=kappa, data=groupsTest)
                envFuncEval = lambda spec=paperSpecEval: makeGroupedEnvironment(spec)
                rlPolicy = makeReinforcementLearningPolicy(agent)

                summary, evalDf = evaluatePolicy(envFuncEval, rlPolicy, episodes=len(groupsTest))
                results[(assetName, *paramCombo)] = (summary["meanCostPct"], summary["stdCostPct"], summary["Y(mean+c*std)"], evalDf)

            print(f"Params: {paramCombo} | "
                  f"meanCostPct: {summary['meanCostPct']:.4f} | "
                  f"stdCostPct: {summary['stdCostPct']:.4f} | "
                  f"Y_metric: {summary['Y(mean+c*std)']:.4f}")

else:
    print("FAST_SUBMISSION_MODE: skipping real-data training/eval.")
    print("Next cell should load final_table_real_data.parquet directly if patched.")

In [ ]:
fastMode = globals().get("FAST_SUBMISSION_MODE", False) or (not globals().get("RUN_HEAVY_REALDATA", True))

if fastMode and os.path.exists("final_table_real_data.parquet"):
    df = pd.read_parquet("final_table_real_data.parquet")
    df
else:
    assert "results" in globals(), ("results not found. Run the real-data training/eval cell first (cell 41), or enable FAST_SUBMISSION_MODE and keep final_table_real_data.parquet in the directory.")
    assert "model_3" in globals(), "`model_3` not found. Run the model-selection cell (cell 32) first."
    assert len(model_3) >= 3, "model_3 must contain at least 3 rows."

    param_map = {i + 1: (float(model_3.iloc[i]["tau"]), float(model_3.iloc[i]["alr"]), float(model_3.iloc[i]["clr"]),
                         float(model_3.iloc[i]["nstd"]), float(model_3.iloc[i]["nclip"]), int(model_3.iloc[i]["rws"]))
                 for i in range(3)}

    # Validate required keys exist in `results`
    requiredKeys = [("paper", 0.01, "delta"), ("paper", 0.05, "delta"), ("stochastic", None, "delta"),
                    ("paper", 0.01, "delta", 0.25), ("paper", 0.05, "delta", 0.25), ("stochastic", None, "delta", 0.25)
                   ]
    for i in [1, 2, 3]:
        requiredKeys += [("paper", 0.01, *param_map[i]), ("paper", 0.05, *param_map[i]), ("stochastic", None, *param_map[i])]

    for key in requiredKeys:
        assert key in results, f"Missing results key: {key}"

    def _fmt_tuple(x):
        return tuple(round(float(v), 1) for v in x)

    # Build compact summary table
    df = pd.DataFrame(index=["Delta", "DeltaWithBands", "Model #1", "Model #2", "Model #3"],
                      columns=["Constant-0.01", "Constant-0.05", "Stochastic"], dtype=object)

    # Baselines
    df.loc["Delta", "Constant-0.01"] = _fmt_tuple(results[("paper", 0.01, "delta")])
    df.loc["Delta", "Constant-0.05"] = _fmt_tuple(results[("paper", 0.05, "delta")])
    df.loc["Delta", "Stochastic"]    = _fmt_tuple(results[("stochastic", None, "delta")])

    df.loc["DeltaWithBands", "Constant-0.01"] = _fmt_tuple(results[("paper", 0.01, "delta", 0.25)])
    df.loc["DeltaWithBands", "Constant-0.05"] = _fmt_tuple(results[("paper", 0.05, "delta", 0.25)])
    df.loc["DeltaWithBands", "Stochastic"]    = _fmt_tuple(results[("stochastic", None, "delta", 0.25)])

    # RL models (top 3)
    for i in [1, 2, 3]:
        df.loc[f"Model #{i}", "Constant-0.01"] = _fmt_tuple(results[("paper", 0.01, *param_map[i])])
        df.loc[f"Model #{i}", "Constant-0.05"] = _fmt_tuple(results[("paper", 0.05, *param_map[i])])
        df.loc[f"Model #{i}", "Stochastic"]    = _fmt_tuple(results[("stochastic", None, *param_map[i])])

    s = df.stack()
    expanded = pd.DataFrame(s.tolist(), index=s.index, columns=["Avg Cost", "Std Cost", "Y Metric"])
    df = expanded.unstack(level=1)
    df.columns = df.columns.swaplevel(0, 1)
    df = df.sort_index(axis=1, level=0)

    # Average Y metric across regimes and sort best-to-worst
    df[("", "Avg Y Metric")] = df.xs("Y Metric", axis=1, level=1).mean(axis=1).round(1)
    df.sort_values(("", "Avg Y Metric"), ascending=True, inplace=True)

    assert not df.isna().any().any(), "Final real-data summary table contains NaNs"

    df.to_parquet("final_table_real_data.parquet")
    df